# waitcnt_101：什么时候要手写 `s_waitcnt`

访存指令发出后不等结果就继续执行，要用 `s_waitcnt` 等它完成。FlyDSL 里可以手写 `fx.rocdl.s_waitcnt(vmcnt=N, lgkmcnt=M)`，但大多数时候编译器已经插好了。这个实验回答：

1. 编译器什么时候自动插等待，插在哪里；
2. 手写的等待和编译器的等待怎样合并，能不能用它"放松"等待；
3. 哪些情况编译器看不到依赖，必须手写（或用 `wait_asyncmark`）；
4. 手写 `vmcnt=N` 时 N 怎么数，容易错在哪里。

ISA 同时编译 gfx942 / gfx950（两者不同时并排打印），GPU 结果在本机 gfx950 上检查。代码里 `# →` 开头的注释就是这一行的实测结果，cell 输出用来核对。

## 0. 背景：两个计数器

每个 wave 有几个计数器，发出一条访存指令加 1，完成时减 1；`s_waitcnt` 等到计数器不大于给定值：

| 计数器 | 统计哪些指令 | 位宽（gfx942 / gfx950） | 完成顺序 |
|---|---|---|---|
| `vmcnt` | global / buffer 的 load 和 store（gfx9 的 store 也算），包括直接写 LDS 的 DMA | 6 位，0..63 | 按发出顺序 |
| `lgkmcnt` | LDS（`ds_*`）、标量 load（`s_load`）、GDS、消息 | 4 位，0..15 | LDS 按顺序；`s_load` 不按顺序，只能等到 0 |
| `expcnt` | export（图形用），计算 kernel 基本不用 | 3 位 | |

因为 `vmcnt` 按发出顺序完成，`vmcnt(N)` 的意思是"除了最近发出的 N 条，前面的都已完成"。`fx.rocdl.s_waitcnt(vmcnt=3)` 只写一个字段时，另外两个取最大值，即不等。

谁来插等待：LLVM 后端最后有一个 pass（`SIInsertWaitcnts`）记录每个寄存器由哪条访存指令写入，在**第一次读这个寄存器之前**插入刚好够用的 `s_waitcnt`。另外，内存模型要求的同步点（如 `gpu.barrier`）也会带上等待。

## 1. 准备

- `make(kfn, block, grid)`：把函数包成 kernel 和 `(X, O, stream)` 的 launcher，每次调用都新建（看 gfx942 的 ISA 和在本机运行要用不同的 launcher，见 [kernel_101](kernel_101.ipynb) 第 3 节）；
- `show(build, *args, fold=True)`：分别编译 gfx942 / gfx950，只打印访存、等待、barrier 和浮点运算；`fold` 把连续的同名指令合并成一行；
- `bad_runs(launcher, args, check, runs, flush)`：重复运行，统计结果错误的次数；`flush=True` 时每次先写一块 256 MB 的 buffer，把 L2 里的数据挤出去。

In [1]:
import os
os.environ["FLYDSL_RUNTIME_ENABLE_CACHE"] = "0"

import contextlib, glob, io, itertools, re, tempfile
from importlib.metadata import version

import torch
import flydsl.compiler as flyc
import flydsl.expr as fx
from flydsl.expr import gpu
from flydsl._mlir import ir
from flydsl._mlir.dialects import llvm

ARCHS = ("gfx942", "gfx950")
LOCAL_ARCH = torch.cuda.get_device_properties(0).gcnArchName.split(":")[0]
stream = fx.Stream(torch.cuda.current_stream().cuda_stream)
print("flydsl", version("flydsl"), "| torch", torch.__version__, "| 本机 GPU", LOCAL_ARCH)

PATTERN = r"global_|buffer_|ds_|s_waitcnt|s_barrier|v_(fma|fmac|add|mul)_f32"


def make(kfn, block=64, grid=1):
    kernel = flyc.kernel(known_block_size=(block, 1, 1))(kfn)

    @flyc.jit
    def launch(X: fx.Tensor, O: fx.Tensor, stream: fx.Stream):
        kernel(X, O).launch(grid=(grid, 1, 1), block=(block, 1, 1), stream=stream)

    return launch


def isa(launcher, *args, arch):
    d = tempfile.mkdtemp(prefix="waitcnt101_")
    env = dict(ARCH=arch, COMPILE_ONLY="1", FLYDSL_DUMP_IR="1", FLYDSL_DUMP_DIR=d)
    old = {k: os.environ.get(k) for k in env}
    os.environ.update(env)
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            flyc.compile(launcher, *args)
    finally:
        for k, v in old.items():
            if v is None:
                os.environ.pop(k)
            else:
                os.environ[k] = v
    path, = glob.glob(f"{d}/**/*final_isa.s", recursive=True)
    return open(path).read()


def body(text):
    lines, inside = [], False
    for raw in text.splitlines():
        if re.match(r"^\w+:", raw):
            inside = True
            continue
        line = raw.split(";")[0].strip()
        if inside and line and (not line.startswith(".") or line.startswith(".LBB")):
            lines.append(line)
        if line.startswith("s_endpgm"):
            break
    return lines


def pick(lines, pattern=PATTERN, fold=True):
    out = []
    for l in (l for l in lines if re.search(pattern, l) or l.startswith(".LBB")):
        op = l.split()[0]
        if fold and out and out[-1][0] == op and not op.startswith(("s_", ".LBB")):
            out[-1][2] += 1
        else:
            out.append([op, l, 1])
    return [l if n == 1 else f"{op} ... × {n}" for op, l, n in out]


def show(build, *args, fold=True, pattern=PATTERN, select=None):
    res = {}
    for arch in ARCHS:
        lines = body(isa(build(), *args, arch=arch))
        res[arch] = pick(select(lines) if select else lines, pattern, fold)
    if res["gfx942"] == res["gfx950"]:
        print("  gfx942 / gfx950 相同：")
        for l in res["gfx942"]:
            print("    " + l)
    else:
        print("  " + " | ".join(f"{a:40}" for a in ARCHS))
        for row in itertools.zip_longest(*res.values(), fillvalue=""):
            print("    " + " | ".join(f"{c:40}" for c in row))


FLUSH = torch.empty(256 * 1024 * 1024 // 4, device="cuda")


def bad_runs(launcher, args, check, runs=1, flush=False):
    bad = 0
    for _ in range(runs):
        args[1].zero_()
        if flush:
            FLUSH.zero_()
        launcher(*args)
        torch.cuda.synchronize()
        bad += not check()
    return bad

flydsl 0.3.4.1 | torch 2.9.1+rocm7.2.0.git7e1940d4 | 本机 GPU gfx950


## 2. 数据进寄存器：编译器自动等，而且等得最晚

4 个 load 连着发，再按顺序用：

In [2]:
def in_order(X: fx.Tensor, O: fx.Tensor):
    i = fx.thread_idx.x
    a0 = X[i]
    a1 = X[i + 64]
    a2 = X[i + 128]
    a3 = X[i + 192]                                   # → 4 条 global_load 连着发出
    O[i] = ((a0 * 2.0 + a1) * 2.0 + a2) * 2.0 + a3    # → 用 a0 前 vmcnt(3)，用 a1 前 vmcnt(2)，… 用 a3 前 vmcnt(0)


X = torch.randn(256, device="cuda")
O = torch.zeros(64, device="cuda")
show(lambda: make(in_order), X, O, stream, fold=False)
print("本机 GPU 正确:", bad_runs(make(in_order), (X, O, stream),
                                lambda: torch.allclose(O, ((X[:64] * 2 + X[64:128]) * 2 + X[128:192]) * 2 + X[192:])) == 0)

  gfx942 / gfx950 相同：
    s_waitcnt lgkmcnt(0)
    global_load_dword v1, v0, s[2:3]
    global_load_dword v2, v0, s[2:3] offset:256
    global_load_dword v3, v0, s[2:3] offset:512
    global_load_dword v4, v0, s[2:3] offset:768
    s_waitcnt vmcnt(3)
    v_add_f32_e32 v1, v1, v1
    s_waitcnt vmcnt(2)
    v_add_f32_e32 v1, v1, v2
    v_add_f32_e32 v1, v1, v1
    s_waitcnt vmcnt(1)
    v_add_f32_e32 v1, v3, v1
    v_add_f32_e32 v1, v1, v1
    s_waitcnt vmcnt(0)
    v_add_f32_e32 v1, v4, v1
    global_store_dword v0, v1, s[4:5]
本机 GPU 正确: True


- 每个值都在第一次被用到之前才等，并且只等到它为止（`a0 * 2.0` 编译成了 `v_add_f32 v1, v1, v1`）。开头的 `lgkmcnt(0)` 是等 `s_load` 读 kernel 参数（标量 load 不按顺序完成，只能等到 0）。
- 这是任何手写等待都做不到更好的位置：再晚就读到旧值，再早就白等。

循环里的预取也一样。下面每轮先发下一轮的 load，再用上一轮预取的值，循环次数是运行时值：

In [3]:
def prefetch(X: fx.Tensor, O: fx.Tensor, n: fx.Int32):
    i = fx.thread_idx.x
    for k, st in range(fx.Int32(0), n, fx.Int32(1), init=[fx.Float32(0.0), X[i]]):
        nxt = X[(fx.Int32(k) + 1) * 64 + i]          # 预取下一轮
        acc = st[0] * 1.5 + st[1]                     # 用上一轮预取的值
        res = yield [acc, nxt]
    O[i] = res[0] + res[1]


def make_prefetch():
    kernel = flyc.kernel(known_block_size=(64, 1, 1))(prefetch)

    @flyc.jit
    def launch(X: fx.Tensor, O: fx.Tensor, n: fx.Int32, stream: fx.Stream):
        kernel(X, O, n).launch(grid=(1, 1, 1), block=(64, 1, 1), stream=stream)

    return launch


def main_loop(lines):
    """第一个有回跳的 label 到回跳指令之间，就是主循环"""
    for i, l in enumerate(lines):
        if l.endswith(":"):
            for j in range(i + 1, len(lines)):
                if lines[j].startswith("s_cbranch") and lines[j].endswith(l[:-1]):
                    return lines[i:j + 1]


X = torch.randn(64 * 16, device="cuda")
O = torch.zeros(64, device="cuda")
# → LLVM 把循环按 8 展开，8 条 load 一起发；用到哪个值才等哪个：vmcnt(6)、vmcnt(7)、vmcnt(6) … vmcnt(1)
show(make_prefetch, X, O, 15, stream, fold=False, select=main_loop,
     pattern=r"global_load|s_waitcnt|v_add_f32|s_cbranch")
ref = torch.zeros(64, device="cuda")
for k in range(15):
    ref = ref * 1.5 + X[k * 64:(k + 1) * 64]
ref = ref + X[15 * 64:]
print("本机 GPU 正确:", bad_runs(make_prefetch(), (X, O, 15, stream), lambda: torch.allclose(O, ref, rtol=1e-5)) == 0)

  gfx942 / gfx950 相同：
    .LBB0_3:
    global_load_dword v10, v[6:7], off offset:256
    global_load_dword v11, v[6:7], off offset:512
    global_load_dword v12, v[6:7], off offset:768
    global_load_dword v13, v[6:7], off offset:1024
    global_load_dword v14, v[6:7], off offset:1280
    global_load_dword v15, v[6:7], off offset:1536
    s_waitcnt vmcnt(6)
    v_add_f32_e32 v4, v4, v1
    global_load_dword v16, v[6:7], off offset:1792
    global_load_dword v1, v[8:9], off
    s_waitcnt vmcnt(7)
    v_add_f32_e32 v4, v4, v10
    s_waitcnt vmcnt(6)
    v_add_f32_e32 v4, v4, v11
    s_waitcnt vmcnt(5)
    v_add_f32_e32 v4, v4, v12
    s_waitcnt vmcnt(4)
    v_add_f32_e32 v4, v4, v13
    s_waitcnt vmcnt(3)
    v_add_f32_e32 v4, v4, v14
    s_waitcnt vmcnt(2)
    v_add_f32_e32 v4, v4, v15
    s_waitcnt vmcnt(1)
    v_add_f32_e32 v4, v4, v16
    s_cbranch_scc1 .LBB0_3
本机 GPU 正确: True


跨迭代的依赖（上一轮发的 load、这一轮用）编译器同样算得出来，没有在循环开头保守地 `vmcnt(0)`。[GEMM README](../kernels/gemm/README.md) 里 v2 主循环的 `vmcnt(13)`～`vmcnt(22)` 也全是编译器自己算的。

**结论：数据经过寄存器的地方（tensor 的 `.load()`、buffer load、`ds_read`、`s_load`、各种返回值的 rocdl intrinsic），不要手写等待。**

## 3. 手写的等待只会更严，不会放松

手写的 `s_waitcnt` 编译器原样保留，再和自己需要的等待合并。所以它只能让等待更早，不能让编译器少等：

In [4]:
def wait_early(X: fx.Tensor, O: fx.Tensor):
    i = fx.thread_idx.x
    a0 = X[i]
    a1 = X[i + 64]
    a2 = X[i + 128]
    a3 = X[i + 192]
    fx.rocdl.s_waitcnt(vmcnt=0)                       # → 保留：4 条全部等完，后面不再有 vmcnt(3/2/1)
    O[i] = ((a0 * 2.0 + a1) * 2.0 + a2) * 2.0 + a3


def wait_loose(X: fx.Tensor, O: fx.Tensor):
    i = fx.thread_idx.x
    a0 = X[i]
    a1 = X[i + 64]
    a2 = X[i + 128]
    a3 = X[i + 192]                                   # → 调度器把 a3 的 load 提到了最前面
    fx.rocdl.s_waitcnt(vmcnt=3)                       # 本意：等到 a0 → 实际只保证 a3（ISA 里第一条）
    O[i] = a3 * 2.0 + a0 + a1 + a2                    # → 编译器照样插 vmcnt(2/1/0)，结果仍然正确


X = torch.randn(256, device="cuda")
O = torch.zeros(64, device="cuda")
print("wait_early:")
show(lambda: make(wait_early), X, O, stream, fold=False)
print("wait_loose:")
show(lambda: make(wait_loose), X, O, stream, fold=False)
print("本机 GPU 正确:", bad_runs(make(wait_loose), (X, O, stream),
                                lambda: torch.allclose(O, X[192:] * 2 + X[:64] + X[64:128] + X[128:192])) == 0)

wait_early:
  gfx942 / gfx950 相同：
    s_waitcnt lgkmcnt(0)
    global_load_dword v1, v0, s[2:3]
    global_load_dword v2, v0, s[2:3] offset:256
    global_load_dword v3, v0, s[2:3] offset:512
    global_load_dword v4, v0, s[2:3] offset:768
    s_waitcnt vmcnt(0)
    v_add_f32_e32 v1, v1, v1
    v_add_f32_e32 v1, v1, v2
    v_add_f32_e32 v1, v1, v1
    v_add_f32_e32 v1, v3, v1
    v_add_f32_e32 v1, v1, v1
    v_add_f32_e32 v1, v4, v1
    global_store_dword v0, v1, s[4:5]
wait_loose:
  gfx942 / gfx950 相同：
    s_waitcnt lgkmcnt(0)
    global_load_dword v1, v0, s[2:3] offset:768
    global_load_dword v2, v0, s[2:3]
    global_load_dword v3, v0, s[2:3] offset:256
    global_load_dword v4, v0, s[2:3] offset:512
    s_waitcnt vmcnt(3)
    v_add_f32_e32 v1, v1, v1
    s_waitcnt vmcnt(2)
    v_add_f32_e32 v1, v2, v1
    s_waitcnt vmcnt(1)
    v_add_f32_e32 v1, v3, v1
    s_waitcnt vmcnt(0)
    v_add_f32_e32 v1, v4, v1
    global_store_dword v0, v1, s[4:5]
本机 GPU 正确: True


两点：

- 想用手写等待"让编译器晚点等"是做不到的；提前写 `vmcnt(0)` 只会让后面的计算等全部 load 回来，失去重叠。
- `vmcnt(N)` 数的是 **ISA 里的发出顺序**，不是源码顺序。`wait_loose` 里调度器把最后一个 load 提到了最前面，按源码数出来的 `vmcnt(3)` 等到的是 `a3` 而不是 `a0`。这里因为数据经过寄存器，编译器又补上了正确的等待；在第 5、6 节编译器看不到依赖的场合，这种数错就是真正的 bug。

## 4. 跨 wave 通信：barrier 前的等待

### 4.1 经过 LDS

256 个线程（4 个 wave）各写 LDS 一个值，barrier 后读下一个 wave 写的值。比较三种 barrier 写法：

In [5]:
@fx.struct
class Smem:
    a: fx.Array[fx.Float32, 256 * 32, 16]


def lds_comm(mode, nper=1):
    def kernel_fn(X: fx.Tensor, O: fx.Tensor):
        tid = fx.thread_idx.x
        base = fx.block_idx.x * 256 * nper
        a = fx.SharedAllocator().allocate(Smem).peek().a
        for j in fx.range_constexpr(nper):
            a[j * 256 + tid] = X[base + j * 256 + tid] * 2.0
        if fx.const_expr(mode == "gpu.barrier"):
            gpu.barrier()                             # → s_waitcnt lgkmcnt(0) + s_barrier
        if fx.const_expr(mode == "s_barrier"):
            fx.rocdl.s_barrier()                      # → 只有 s_barrier，ds_write 可能还没完成
        if fx.const_expr(mode == "waitcnt + s_barrier"):
            fx.rocdl.s_waitcnt(lgkmcnt=0)             # → 手写补上，和 gpu.barrier 一样
            fx.rocdl.s_barrier()
        for j in fx.range_constexpr(nper):
            O[base + j * 256 + tid] = a[j * 256 + (tid + 64) % 256]
    return kernel_fn


X = torch.randn(256, device="cuda")
O = torch.zeros(256, device="cuda")
for mode in ["gpu.barrier", "s_barrier", "waitcnt + s_barrier"]:
    print(mode + ":")
    show(lambda: make(lds_comm(mode), 256), X, O, stream)

# 压力测试：1024 个 workgroup，每个线程写 32 个值
G, NPER = 1024, 32
X = torch.randn(G * 256 * NPER, device="cuda")
O = torch.zeros_like(X)
expect = (X * 2).view(G, NPER, 256).roll(-64, dims=2).flatten()
for mode in ["gpu.barrier", "s_barrier"]:
    n = bad_runs(make(lds_comm(mode, NPER), 256, G), (X, O, stream), lambda: torch.equal(O, expect), runs=200)
    print(f"{mode:12} 本机 GPU 错误次数: {n} / 200")         # → 两种都是 0

gpu.barrier:
  gfx942 / gfx950 相同：
    s_waitcnt lgkmcnt(0)
    global_load_dword v1, v[4:5], off
    s_waitcnt vmcnt(0)
    v_add_f32_e32 v1, v1, v1
    ds_write_b32 v5, v1
    s_waitcnt lgkmcnt(0)
    s_barrier
    ds_read_b32 v4, v0
    s_waitcnt lgkmcnt(0)
    global_store_dword v[0:1], v4, off
s_barrier:
  gfx942 / gfx950 相同：
    s_waitcnt lgkmcnt(0)
    global_load_dword v1, v[4:5], off
    s_waitcnt vmcnt(0)
    v_add_f32_e32 v1, v1, v1
    ds_write_b32 v5, v1
    s_barrier
    ds_read_b32 v4, v0
    s_waitcnt lgkmcnt(0)
    global_store_dword v[0:1], v4, off
waitcnt + s_barrier:
  gfx942 / gfx950 相同：
    s_waitcnt lgkmcnt(0)
    global_load_dword v1, v[4:5], off
    s_waitcnt vmcnt(0)
    v_add_f32_e32 v1, v1, v1
    ds_write_b32 v5, v1
    s_waitcnt lgkmcnt(0)
    s_barrier
    ds_read_b32 v4, v0
    s_waitcnt lgkmcnt(0)
    global_store_dword v[0:1], v4, off
gpu.barrier  本机 GPU 错误次数: 0 / 200
s_barrier    本机 GPU 错误次数: 0 / 200


- `gpu.barrier()` 是带内存语义的 barrier（workgroup 范围的 release/acquire fence + `s_barrier`），LLVM 按内存模型在前面插了 `lgkmcnt(0)`，保证本 wave 的 LDS 写已完成。
- `fx.rocdl.s_barrier()` 只是一条裸指令，编译器不加任何等待。要自己写 `s_waitcnt(lgkmcnt=0)`。
- 本机压力测试里裸 `s_barrier` 没测出错误（同一个 CU 的 LDS 请求实际上基本按顺序处理），但内存模型不保证这一点，`lgkmcnt(0)` 也几乎没有代价。**直接用 `gpu.barrier()`，不要手写。**

### 4.2 经过 global memory（同一个 workgroup 内）

同样的通信改成经过 global：先 `O[tid] = ...`，barrier 后读 `O[(tid + 64) % 256]`：

In [6]:
def global_comm(mode):
    def kernel_fn(X: fx.Tensor, O: fx.Tensor):
        tid = fx.thread_idx.x
        O[tid] = X[tid] * 2.0
        if fx.const_expr(mode == "gpu.barrier"):
            gpu.barrier()                             # → 只有 s_barrier，没有 vmcnt(0)
        if fx.const_expr(mode == "waitcnt + gpu.barrier"):
            fx.rocdl.s_waitcnt(vmcnt=0)               # → 手写的 vmcnt(0) 保留
            gpu.barrier()
        O[tid + 256] = O[(tid + 64) % 256]
    return kernel_fn


X = torch.randn(256, device="cuda")
O = torch.zeros(512, device="cuda")
for mode in ["gpu.barrier", "waitcnt + gpu.barrier"]:
    print(mode + ":")
    show(lambda: make(global_comm(mode), 256), X, O, stream)
    n = bad_runs(make(global_comm(mode), 256), (X, O, stream), lambda: torch.equal(O[256:], (X * 2).roll(-64)), runs=2000)
    print(f"  本机 GPU 错误次数: {n} / 2000")           # → 两种都是 0

gpu.barrier:
  gfx942 / gfx950 相同：
    s_waitcnt lgkmcnt(0)
    global_load_dword v2, v1, s[2:3]
    s_waitcnt vmcnt(0)
    v_add_f32_e32 v2, v2, v2
    global_store_dword v1, v2, s[4:5]
    s_barrier
    global_load_dword v0, v0, s[4:5]
    s_waitcnt vmcnt(0)
    global_store_dword v1, v0, s[4:5] offset:1024
  本机 GPU 错误次数: 0 / 2000
waitcnt + gpu.barrier:
  gfx942 / gfx950 相同：
    s_waitcnt lgkmcnt(0)
    global_load_dword v2, v1, s[2:3]
    s_waitcnt vmcnt(0)
    v_add_f32_e32 v2, v2, v2
    global_store_dword v1, v2, s[4:5]
    s_waitcnt vmcnt(0)
    s_barrier
    global_load_dword v0, v0, s[4:5]
    s_waitcnt vmcnt(0)
    global_store_dword v1, v0, s[4:5] offset:1024
  本机 GPU 错误次数: 0 / 2000


`gpu.barrier()` 在 global store 之后**没有**插 `vmcnt(0)`：它的 fence 是 workgroup 范围，同一个 workgroup 的 wave 在同一个 CU 上，LLVM 的 gfx942 / gfx950 内存模型认为这时不需要等 store 完成，本机 2000 次结果也都正确。所以 workgroup 内经过 global 通信也不用手写。跨 workgroup 的通信不在 barrier 的范围内，要用 atomic 和更大范围的 fence，不是 `s_waitcnt` 能解决的。

## 5. 编译器看不到的写：DMA 到 LDS

`buffer_load ... lds` 把数据直接写进 LDS，不经过寄存器（见 [LDS_101](lds_101.ipynb) 第 7 节）。FlyDSL 的两类 copy atom 对等待的处理不同：

- `fx.rocdl.BufferCopyLDS`：同步 atom，编译器把它当成 LDS 写来跟踪，读 LDS 前自动插 `vmcnt`；
- `fx.rocdl.cdna4.BufferLoadAsyncLDS` / `GlobalLoadAsyncLDS`：异步 atom，编译器**不跟踪**，等待要自己写：`fx.rocdl.asyncmark()` 把之前发出的 DMA 结成一组，`fx.rocdl.wait_asyncmark(n)` 等到最多剩 n 组未完成；或者直接写 `s_waitcnt(vmcnt=...)`。

每个线程发 16 条 4 B 的 DMA，barrier 后读另一个 wave 搬的数据：

In [7]:
@fx.struct
class Tile:
    a: fx.Array[fx.Float32, 256 * 16, 16]


def dma(mode, nper=16):
    def kernel_fn(X: fx.Tensor, O: fx.Tensor):
        tid = fx.thread_idx.x
        lds = fx.SharedAllocator().allocate(Tile).peek()
        buf = fx.rocdl.make_buffer_tensor(fx.make_view(fx.get_iter(X), fx.make_layout(256 * nper, 1)))
        op = fx.rocdl.BufferCopyLDS(32) if mode == "sync" else fx.rocdl.cdna4.BufferLoadAsyncLDS(32)
        atom = fx.make_copy_atom(op, 32)
        for j in fx.range_constexpr(nper):
            off = j * 256 + tid
            fx.copy(atom, fx.make_view(fx.add_offset(fx.get_iter(buf), off), fx.make_layout(1, 1)),
                    fx.make_view(fx.add_offset(lds.a.ptr, off), fx.make_layout(1, 1)))
        if fx.const_expr(mode == "async + wait_asyncmark"):
            fx.rocdl.asyncmark()
            fx.rocdl.wait_asyncmark(0)                # → s_waitcnt vmcnt(0)
        if fx.const_expr(mode == "async + s_waitcnt"):
            fx.rocdl.s_waitcnt(vmcnt=0)               # → 同上
        gpu.barrier()                                 # sync → 编译器在这里插 vmcnt(0)；async 不写等待 → 没有 vmcnt
        for j in fx.range_constexpr(nper):
            O[j * 256 + tid] = lds.a[j * 256 + (tid + 64) % 256]
    return kernel_fn


X = torch.randn(256 * 16, device="cuda")
O = torch.zeros_like(X)
expect = X.view(16, 256).roll(-64, dims=1).flatten()
for mode in ["sync", "async", "async + wait_asyncmark", "async + s_waitcnt"]:
    print(mode + ":")
    show(lambda: make(dma(mode), 256), X, O, stream, pattern=r" lds$|s_waitcnt.*vmcnt|s_barrier")
    n = bad_runs(make(dma(mode), 256), (X, O, stream), lambda: torch.equal(O, expect), runs=200, flush=True)
    print(f"  本机 GPU 错误次数: {n} / 200")             # → 只有不写等待的 async 出错（约一半的次数）

sync:
  gfx942 / gfx950 相同：
    buffer_load_dword ... × 16
    s_waitcnt vmcnt(0)
    s_barrier
  本机 GPU 错误次数: 0 / 200
async:
  gfx942 / gfx950 相同：
    buffer_load_dword ... × 16
    s_barrier
  本机 GPU 错误次数: 96 / 200
async + wait_asyncmark:
  gfx942 / gfx950 相同：
    buffer_load_dword ... × 16
    s_waitcnt vmcnt(0)
    s_barrier
  本机 GPU 错误次数: 0 / 200
async + s_waitcnt:
  gfx942 / gfx950 相同：
    buffer_load_dword ... × 16
    s_waitcnt vmcnt(0)
    s_barrier
  本机 GPU 错误次数: 0 / 200


不写等待的 async 版本在 barrier 前没有 `vmcnt`，数据从 HBM 读（`flush=True`）时大约一半的运行结果是错的。同步 atom 不需要手写；异步 atom 必须自己等。

### 5.1 手数 `vmcnt` 的坑

异步 DMA 的意义是"只等需要的那组，后面的继续在飞"。两组 DMA（每组每线程 4 条），中间再读一个普通值 `v`，读第 0 组前想让第 1 组和 `v` 继续在飞：

- `wait_asyncmark(1)`：最多剩 1 组；
- 手写 `s_waitcnt(vmcnt=K + 1)`：按源码顺序，第 0 组之后还有 K 条 DMA + 1 条 load。

In [8]:
K = 4


@fx.struct
class Tile2:
    a: fx.Array[fx.Float32, 256 * 2 * K, 16]


def two_groups(mode):
    def kernel_fn(X: fx.Tensor, O: fx.Tensor):
        tid = fx.thread_idx.x
        lds = fx.SharedAllocator().allocate(Tile2).peek()
        buf = fx.rocdl.make_buffer_tensor(fx.make_view(fx.get_iter(X), fx.make_layout(256 * 2 * K, 1)))
        atom = fx.make_copy_atom(fx.rocdl.cdna4.BufferLoadAsyncLDS(32), 32)
        for h in fx.range_constexpr(2):
            for j in fx.range_constexpr(K):
                off = (h * K + j) * 256 + tid
                fx.copy(atom, fx.make_view(fx.add_offset(fx.get_iter(buf), off), fx.make_layout(1, 1)),
                        fx.make_view(fx.add_offset(lds.a.ptr, off), fx.make_layout(1, 1)))
            if fx.const_expr(mode == "wait_asyncmark"):
                fx.rocdl.asyncmark()
        v = X[(tid * 7) % 256]
        # wait_asyncmark → 顺序不变：8 条 DMA、1 条 load，vmcnt(5) = 第 1 组 4 条 + load，正确
        # s_waitcnt      → load 被调度到第 0 组中间：3 条 DMA、load、5 条 DMA；vmcnt(5) 只保证前 4 条，
        #                  第 0 组的第 4 条 DMA 不保证完成
        if fx.const_expr(mode == "wait_asyncmark"):
            fx.rocdl.wait_asyncmark(1)
        else:
            fx.rocdl.s_waitcnt(vmcnt=K + 1)
        gpu.barrier()
        for j in fx.range_constexpr(K):
            O[j * 256 + tid] = lds.a[j * 256 + (tid + 64) % 256] + v
    return kernel_fn


X = torch.randn(256 * 2 * K, device="cuda")
O = torch.zeros(256 * K, device="cuda")
expect = X[:256 * K].view(K, 256).roll(-64, dims=1) + X[(torch.arange(256, device="cuda") * 7) % 256]
for mode in ["wait_asyncmark", "s_waitcnt"]:
    print(mode + ":")
    show(lambda: make(two_groups(mode), 256), X, O, stream, pattern=r" lds$|global_load|s_waitcnt.*vmcnt|s_barrier")
    n = bad_runs(make(two_groups(mode), 256), (X, O, stream), lambda: torch.equal(O.view(K, 256), expect), runs=500, flush=True)
    print(f"  本机 GPU 错误次数: {n} / 500")             # → 都是 0：手数的版本有隐患，但这次没测出来

wait_asyncmark:
  gfx942 / gfx950 相同：
    buffer_load_dword ... × 8
    global_load_dword v4, v1, s[6:7]
    s_waitcnt vmcnt(5) lgkmcnt(0)
    s_barrier
    s_waitcnt vmcnt(0) lgkmcnt(2)
  本机 GPU 错误次数: 0 / 500
s_waitcnt:
  gfx942 / gfx950 相同：
    buffer_load_dword ... × 3
    global_load_dword v3, v3, s[4:5]
    buffer_load_dword ... × 5
    s_waitcnt vmcnt(5)
    s_barrier
  本机 GPU 错误次数: 0 / 500


- 普通 load 和异步 DMA 之间没有依赖，调度器可以随意移动它。手写的 `vmcnt(5)` 是按源码顺序数的，ISA 顺序一变就少等了一条——第 0 组的最后一条 DMA 可能还没写进 LDS。本机 500 次没有测出错误（这条 DMA 紧跟在 load 后面发出，差不多同时完成），但这是一个随时可能出现的 race。
- 用 `asyncmark` 的版本里，load 没有被移进 DMA 组中间；而且 `wait_asyncmark(n)` 是编译器在最后插等待时按实际 ISA 顺序换算成 `vmcnt` 的，会把中间的普通 load / store 都算进去（[LDS_101](lds_101.ipynb) 7.3 节里 `wait_asyncmark(0)` 变成了 `vmcnt(2)`，就是因为中间有 2 条 store）。
- **异步 DMA 优先用 `asyncmark` / `wait_asyncmark`，不要手数 `vmcnt`。**

## 6. 编译器看不到的读：inline asm 发出的访存

`llvm.inline_asm` 发出的指令编译器不认识，它不知道 asm 的输出寄存器要等 `vmcnt`：

In [9]:
def asm_load(wait):
    def kernel_fn(X: fx.Tensor, O: fx.Tensor, addr: fx.Int64):
        tid = fx.thread_idx.x
        p = addr + fx.Int64(tid) * 4
        v = llvm.inline_asm(ir.F32Type.get(), [p.ir_value()], "global_load_dword $0, $1, off", "=v,v",
                            has_side_effects=True)
        if fx.const_expr(wait):
            fx.rocdl.s_waitcnt(vmcnt=0)               # → 手写的等待；不写则 load 后直接 v_add
        O[tid] = fx.Float32(v) * 2.0
    return kernel_fn


def make_asm(kfn):
    kernel = flyc.kernel(known_block_size=(64, 1, 1))(kfn)

    @flyc.jit
    def launch(X: fx.Tensor, O: fx.Tensor, addr: fx.Int64, stream: fx.Stream):
        kernel(X, O, addr).launch(grid=(1, 1, 1), block=(64, 1, 1), stream=stream)

    return launch


X = torch.randn(64, device="cuda")
O = torch.zeros(64, device="cuda")
for wait in [False, True]:
    print(f"wait={wait}:")
    show(lambda: make_asm(asm_load(wait)), X, O, X.data_ptr(), stream)
    O.zero_()
    make_asm(asm_load(wait))(X, O, X.data_ptr(), stream)
    torch.cuda.synchronize()
    print(f"  本机 GPU 正确的元素: {(O == X * 2).sum().item()} / 64")   # → 不等：0 / 64；等：64 / 64

wait=False:
  gfx942 / gfx950 相同：
    s_waitcnt lgkmcnt(0)
    global_load_dword v1, v[2:3], off
    v_add_f32_e32 v1, v1, v1
    global_store_dword v0, v1, s[4:5]
  本机 GPU 正确的元素: 0 / 64
wait=True:
  gfx942 / gfx950 相同：
    s_waitcnt lgkmcnt(0)
    global_load_dword v1, v[2:3], off
    s_waitcnt vmcnt(0)
    v_add_f32_e32 v1, v1, v1
    global_store_dword v0, v1, s[4:5]
  本机 GPU 正确的元素: 64 / 64


不写等待时，`v_add` 读到的是 load 还没写回的旧寄存器，64 个结果全错。用 inline asm 发访存（例如 pyhip 风格的手写流水），等待就要全部自己写；`vmcnt` 的计数要包括编译器自己发出的 load / store，而它们的位置不受控制（第 5.1 节），所以要么写 `vmcnt(0)`，要么把这一段的访存全部写进 asm。

## 总结

| 场景 | 编译器自动等吗 | 要不要手写 |
|---|---|---|
| 数据进寄存器：`.load()`、buffer load、`ds_read`、`s_load` | 是，在第一次使用前、只等到需要的那条；循环跨迭代也算得对 | 不要 |
| 想让编译器晚点等 | 手写的等待原样保留并和编译器的合并，只会更严 | 做不到，提前写反而失去重叠 |
| 经过 LDS 跨 wave 通信 | `gpu.barrier()` 自带 `lgkmcnt(0)`；裸 `fx.rocdl.s_barrier()` 不带 | 用 `gpu.barrier()`；用裸 `s_barrier` 时手写 `lgkmcnt=0` |
| workgroup 内经过 global 通信 | `gpu.barrier()` 不插 `vmcnt(0)`，按内存模型已足够 | 不要；跨 workgroup 用 atomic + fence |
| 同步 DMA 到 LDS（`BufferCopyLDS`） | 是，读 LDS 前 `vmcnt` | 不要 |
| 异步 DMA 到 LDS（`BufferLoadAsyncLDS` / `GlobalLoadAsyncLDS`） | 否，不写就读到旧数据 | 必须：`asyncmark` + `wait_asyncmark(n)`；手写 `vmcnt=N` 会因调度顺序数错 |
| inline asm 访存 | 否 | 必须手写；计数要包括编译器自己的访存，稳妥做法是 `vmcnt(0)` |

`vmcnt(N)` 的 N 永远按 **ISA 里的发出顺序**数，包括 store。只要数据经过寄存器、或者用了 FlyDSL 的同步原语，就交给编译器。